# Import libraries

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from pandas_gbq import to_gbq
import os

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Set the environment variable for Google Cloud credentials
# Place the path in which the .json file is located.

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"C:\Users\roxan\AppData\Roaming\gcloud\application_default_credentials.json"

In [3]:
# Set your Google Cloud project ID and BigQuery dataset details

project_id = 'project-401f4646-3663-4125-aaa' # Edit with your project id
dataset_id = 'staging_db' # Modify the necessary schema name: staging_db, reporting_db etc.
table_id = 'stg_staff' # Modify the necessary table name: stg_customer, stg_city etc.

# SQL Query

In [4]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Define your SQL query here
query = """
with base as (
  select *
  from `project-401f4646-3663-4125-aaa.pagila_productionpublic.staff` 
  )

  , final as (
    select
          staff_id
        , first_name as staff_first_name
        , last_name as staff_last_name
        , address_id as staff_address_id
        , email as staff_email
        , store_id as staff_store_id
        , active as staff_active
        , username as staff_username
        , password as staff_password
        , last_update as staff_last_update
        , picture as staff_picture
   FROM base
  )

  select * from final
"""

# Execute the query and store the result in a dataframe
df = client.query(query).to_dataframe()

# Explore some records
df.head()

C:\Users\roxan\PycharmProjects\Pagila Analytics Pipeline\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,staff_id,staff_first_name,staff_last_name,staff_address_id,staff_email,staff_store_id,staff_active,staff_username,staff_password,staff_last_update,staff_picture
0,1,Mike,Hillyer,3,Mike.Hillyer@sakilastaff.com,1,True,Mike,8cb2237d0679ca88db6464eac60da96345513964,2022-05-16 15:13:11.793280+00:00,b'\x89PNG\r\nZ\n'
1,2,Jon,Stephens,4,Jon.Stephens@sakilastaff.com,2,True,Jon,8cb2237d0679ca88db6464eac60da96345513964,2022-05-16 15:13:11.793280+00:00,None


# Write to BigQuery

In [5]:
# Define the full table ID
full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# Define table schema based on the project description

schema = [
    bigquery.SchemaField('staff_id', 'INTEGER'),
    bigquery.SchemaField('staff_first_name', 'STRING'),
    bigquery.SchemaField('staff_last_name', 'STRING'),
    bigquery.SchemaField('staff_address_id', 'INTEGER'),
    bigquery.SchemaField('staff_email', 'STRING'),
    bigquery.SchemaField('staff_store_id', 'INTEGER'),
    bigquery.SchemaField('staff_active', 'INTEGER'),
    bigquery.SchemaField('staff_username', 'STRING'),
    bigquery.SchemaField('staff_password', 'STRING'),
    bigquery.SchemaField('staff_last_update', 'DATETIME'),
    bigquery.SchemaField('staff_picture', 'BYTES'),

    ]

In [6]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Check if the table exists
def table_exists(client, full_table_id):
    try:
        client.get_table(full_table_id)
        return True
    except Exception:
        return False

# Write the dataframe to the table (overwrite if it exists, create if it doesn't)
if table_exists(client, full_table_id):
    # If the table exists, overwrite it
    destination_table = f"{dataset_id}.{table_id}"
    # Write the dataframe to the table (overwrite if it exists)
    to_gbq(df, destination_table, project_id=project_id, if_exists='replace')
    print(f"Table {full_table_id} exists. Overwritten.")
else:
    # If the table does not exist, create it
    job_config = bigquery.LoadJobConfig(schema=schema)
    job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
    job.result()  # Wait for the job to complete
    print(f"Table {full_table_id} did not exist. Created and data loaded.")

Table project-401f4646-3663-4125-aaa.staging_db.stg_staff did not exist. Created and data loaded.


In [7]:
# Converting i.pynb file to .py python executable file. 

!python -m jupyter nbconvert stg_staff.ipynb --to python --output-dir='../'

[NbConvertApp] Converting notebook stg_staff.ipynb to python
[NbConvertApp] Writing 3744 bytes to ..\stg_staff.py


In [8]:
!python -m pip install nbconvert


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!python -m pip install nbconvert -U


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
